# 01 — Colab setup for the FER project

Run cells top-to-bottom on a fresh Colab session.

Steps:
1. Mount Google Drive (datasets and checkpoints persist there across session restarts).
2. Clone this repo into `/content/fer` (replace the URL with your fork once you push).
3. Install Python deps from `requirements.txt`.
4. Unpack `fer2013.zip` and (if EULA approved) `rafdb.zip` from Drive.
5. Run the dataset prep scripts.
6. Smoke-test one batch (4×4 grid).

Reference report: `research/facial-emotion-recognition/report.md`.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repo and install deps

Replace `REPO_URL` with your remote once you push. For now, set it to your local path or upload a tarball to Drive.

In [ ]:
# Clone the FER repo. Token comes from Colab Secrets (sidebar -> key icon -> GH_TOKEN).
import os
from google.colab import userdata
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
!cd /content && rm -rf fer && git clone https://$GH_TOKEN@github.com/radudeaconu/fer.git fer
%cd /content/fer
!pip install -q -r requirements.txt

# Set up the Drive workspace and link runs/ now so anything written below persists.
%run scripts/colab_bootstrap.py


## 3. Unpack datasets from Drive

Place `fer2013.zip` (and optionally `rafdb.zip`) under `MyDrive/fer-data/` before running.

In [ ]:
import os, shutil, zipfile
from pathlib import Path

DATA_ROOT = Path('/content/fer/data')
DATA_ROOT.mkdir(exist_ok=True)
DRIVE_DATA = Path('/content/drive/MyDrive/fer-data')

# Accept either fer2013.zip OR a bare fer2013.csv on Drive (whichever is faster for you to upload).
fer_dest = DATA_ROOT / 'fer2013'
fer_dest.mkdir(exist_ok=True)
if (DRIVE_DATA / 'fer2013.zip').exists() and not (fer_dest / 'fer2013.csv').exists():
    with zipfile.ZipFile(DRIVE_DATA / 'fer2013.zip') as z:
        z.extractall(fer_dest)
    print(f'Unpacked fer2013.zip -> {fer_dest}')
elif (DRIVE_DATA / 'fer2013.csv').exists() and not (fer_dest / 'fer2013.csv').exists():
    shutil.copy(DRIVE_DATA / 'fer2013.csv', fer_dest / 'fer2013.csv')
    print(f'Copied fer2013.csv -> {fer_dest}')
elif (fer_dest / 'fer2013.csv').exists():
    print(f'FER-2013 CSV already at {fer_dest}')
else:
    print(f'WARNING: neither {DRIVE_DATA}/fer2013.zip nor {DRIVE_DATA}/fer2013.csv found.')

# RAF-DB stays optional (EULA-gated).
if (DRIVE_DATA / 'rafdb.zip').exists():
    raf_dest = DATA_ROOT / 'rafdb'
    raf_dest.mkdir(exist_ok=True)
    with zipfile.ZipFile(DRIVE_DATA / 'rafdb.zip') as z:
        z.extractall(raf_dest)
    print(f'Unpacked rafdb.zip -> {raf_dest}')
else:
    print('Skipping RAF-DB (EULA pending).')

## 4. Run dataset prep scripts

In [ ]:
if (DATA_ROOT / 'fer2013' / 'fer2013.csv').exists():
    !python scripts/prepare_fer2013.py --csv data/fer2013/fer2013.csv --out data/fer2013
else:
    print('FER-2013 CSV not found; skipping prep.')

if (DATA_ROOT / 'rafdb' / 'EmoLabel' / 'list_patition_label.txt').exists():
    !python scripts/prepare_rafdb.py --root data/rafdb
else:
    print('RAF-DB labels not found; skipping prep (waiting on EULA).')

## 4b. Publish prepared data to Drive

Tar up `data/fer2013/` and stage it at `MyDrive/fer-workspace/data_archives/fer2013_prepared.tar.gz`. Subsequent notebooks pull from this archive on session start (much faster than re-running `prepare_fer2013.py` or reading 36k PNGs over Drive FUSE).


In [ ]:
!python scripts/colab_publish_data.py


## 5. Run smoke tests

In [ ]:
!python -m pytest tests/ -v

## 6. Visualize a batch (4×4 grid)

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from src.data import CLASSES, FER2013Dataset, RAFDBDataset, build_transforms

DATASET = 'fer2013'  # or 'rafdb' once EULA approved
Cls = FER2013Dataset if DATASET == 'fer2013' else RAFDBDataset

ds = Cls(f'data/{DATASET}', split='train', transform=build_transforms(train=False, image_size=112))
loader = DataLoader(ds, batch_size=16, shuffle=True)
imgs, labels = next(iter(loader))

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
for ax, img, lbl in zip(axes.flat, imgs, labels):
    img = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img); ax.axis('off')
    ax.set_title(CLASSES[lbl.item()], fontsize=10)
plt.tight_layout()
plt.show()